# Chapter 9: More on Specification and Data Issues

Topics: functional form misspecification, measurement error, missing data, outliers, LAD estimation.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import OLSInfluence
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 9.1 Functional Form Misspecification (RESET Test)

## Step 2 – RESET Test for Functional Form Misspecification

**Ramsey RESET procedure:**
1. Estimate original model → get $\hat{y}$
2. Add $\hat{y}^2, \hat{y}^3$ to the regression
3. F-test H₀: coefficients on $\hat{y}^2, \hat{y}^3$ are jointly zero

Rejection → functional form problem (try log transformation or polynomial terms).

In [ ]:
df = woo.dataWoo('hprice1')
model = smf.ols('price ~ lotsize + sqrft + bdrms', data=df).fit()

from statsmodels.stats.diagnostic import linear_reset
reset = linear_reset(model, power=2, use_f=True)
print(f"RESET F-stat: {reset.statistic:.4f}, p-value: {reset.pvalue:.4f}")

# Log model
model_log = smf.ols('np.log(price) ~ np.log(lotsize) + np.log(sqrft) + bdrms', data=df).fit()
reset_log = linear_reset(model_log, power=2, use_f=True)
print(f"\nLog model RESET: F-stat={reset_log.statistic:.4f}, p-value={reset_log.pvalue:.4f}")

## 9.4 Outlying Observations

## Step 3 – Measurement Error

**Error in the dependent variable** ($y^* = y + e_0$): OLS remains unbiased but efficiency is reduced (larger variance).

**Error in a regressor** ($x_1 = x_1^* + e_1$, where $x_1^*$ is the true value):
$$\text{plim}(\hat{\beta}_1) = \beta_1 \cdot \frac{\sigma_{x_1^*}^2}{\sigma_{x_1^*}^2 + \sigma_{e_1}^2} \neq \beta_1$$

This is **attenuation bias** — the estimated coefficient is biased towards zero. The bias factor is the **reliability ratio** $\lambda = \sigma_{x^*}^2/(\sigma_{x^*}^2+\sigma_e^2) < 1$.

In [ ]:
df = woo.dataWoo('rdchem')
model = smf.ols('rdintens ~ sales + I(sales**2) + profmarg', data=df).fit()

influence = OLSInfluence(model)
stud_resid = influence.resid_studentized_external
leverage   = influence.hat_matrix_diag
cooks_d    = influence.cooks_distance[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(leverage, stud_resid, alpha=0.6, color='steelblue')
axes[0].axhline(2,  color='red', linestyle='--', label='±2')
axes[0].axhline(-2, color='red', linestyle='--')
axes[0].set_xlabel('Leverage'); axes[0].set_ylabel('Studentized Residuals')
axes[0].set_title("Leverage vs Studentized Residuals")
axes[0].legend()

axes[1].stem(cooks_d, markerfmt='C1o', linefmt='C1-', basefmt='gray')
axes[1].axhline(4/len(df), color='red', linestyle='--', label=f"4/n = {4/len(df):.3f}")
axes[1].set_title("Cook's Distance"); axes[1].legend()

plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_outliers.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_09_Specification_Data_Issues_01.png'), dpi=150, bbox_inches='tight')
plt.show()

outliers = np.where(np.abs(stud_resid) > 2)[0]
print(f"Potential outliers (|stud resid| > 2): observations {outliers}")

## Step 4 – Influential Observations and Outliers

**Leverage** measures how far $\mathbf{x}_i$ is from the mean of $\mathbf{X}$:
$$h_{ii} = \mathbf{x}_i'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_i \in [0,1]$$

High leverage: $h_{ii} > 2(k+1)/n$ — the observation can have a large influence.

**Studentised residual** (externally studentised):
$$r_i = \frac{\hat{u}_i}{\hat{\sigma}_{(i)}\sqrt{1-h_{ii}}}$$

Outlier if $|r_i| > 2$ (approximately).

**Cook's distance** combines leverage and residual size:
$$D_i = \frac{r_i^2}{k+1} \cdot \frac{h_{ii}}{1-h_{ii}}$$

Rule of thumb: $D_i > 1$ (or $> 4/n$) flags influential observations.